## 1. Setup and Configuration

### 1.1 Environment Setup

#### 1.1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install required packages
!pip install -qU pip setuptools wheel
!pip install --upgrade transformers datasets evaluate rouge-score

# 4) Make result directories
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction").mkdir(parents=True, exist_ok=True)

# 5) GPU check
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 6) Set canonical task name for this notebook
TASK_NAME = "location-extraction"

Mounted at /content/drive


#### 1.1.2 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# import os

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# os.makedirs(RESULTS_PATH, exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# print("✅ Local setup complete!")

### 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import sys
import pandas as pd
import numpy as np
from collections import defaultdict
import re

# Machine learning and transformers
import torch
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration,
    Trainer, 
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset

# Evaluation metrics
import evaluate
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File I/O utilities (for saving results consistently)
sys.path.append('/content/code-satp/models/location-models')
try:
    from utils.file_io import (
        save_dataframe_csv, 
        load_dataframe_csv,
        get_task_results_dir, 
        build_filename, 
        normalize_task_name
    )
    print("✅ File I/O utilities loaded successfully")
except ImportError:
    print("⚠️ File I/O utilities not available (working locally)")

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Data Loading and Preparation

### 2.1 Load Raw Data

In [ ]:
# Try loading from cloned repository first
try:
    data = pd.read_csv('/content/code-satp/data/location_info.csv', dtype=str)
    print("✅ Data loaded successfully from cloned repository")
    print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
except Exception as e:
    print(f"❌ Error loading CSV from cloned repo: {e}")
    print("🔧 Fallback: Trying GitHub URL...")
    
    # Fallback to GitHub if cloned repo fails or if working locally
    url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info.csv"
    try:
        data = pd.read_csv(url, dtype=str)
        print("✅ Data loaded successfully from GitHub")
        print(f"Rows: {len(data)}, Columns: {len(data.columns)}")
    except Exception as e:
        print(f"❌ Error loading CSV from GitHub: {e}")

,incident_number,state,district,block,village_name,other_areas,constituency,incident_summary,Matched_Location
0,101010701,Andhra Pradesh,Hyderabad,Gachibowli (Rangareddy),NaN,Cyberabad,Serilingampally,An alleged arms supplier to the Communist Part...,"hyderabad, cyberabad"
1,101010901,Andhra Pradesh,Nizamabad,NaN,Kamareddy,NaN,Kamareddy,A Kamareddy dalam (squad) member belonging to ...,"nizamabad, kamareddy, kamareddy"
2,101030601,Andhra Pradesh,Khammam,NaN,Bhadrachalam,NaN,Bhadrachalam,Senior CPI-Maoist 'Polit Bureau' and 'central ...,"khammam, bhadrachalam, bhadrachalam"
3,101051602,Andhra Pradesh,Vishakhapatnam,NaN,NaN,Visakha Agency,Rayadurg,A TDP leader and former Sarpanch of Jerrela Gr...,"vishakhapatnam, visakha agency"
4,101060701,Andhra Pradesh,Visakhapatnam,GK Veedhi,Teegalabanda,Pedalavasa,Rayadurg,The CPI-Maoist cadres blasted coffee pulping u...,"veedhi, teegalabanda, pedavalasa"


### 2.2 Create Structured Location Labels

#### About Structured Output Format

This notebook trains a T5-base model to extract location information from incident summaries using a **structured output format** with explicit labels for each administrative level.

**Key Changes from Previous Approach:**

1. **Data Source**
   - Using `location_info.csv` which contains human-annotated location data
   - Ground truth comes from separate columns: `state`, `district`, `block`, `village_name`
   - No longer relying on NLP-extracted `Matched_Location` column

2. **Structured Output Format**
   - **Old Format:** `"Chhattisgarh, Sukma, Nilamadgu"` (ambiguous)
   - **New Format:** `"state: Chhattisgarh, district: Sukma, village: Nilamadgu"` (explicit)
   
   **Benefits:**
   - Clear hierarchy with labeled levels
   - Easier to parse and validate
   - Better training signal for the model
   - Handles missing levels gracefully
   - Enables level-specific evaluation metrics

3. **Enhanced Evaluation Metrics**
   Instead of simple token matching, we now compute:
   - **Exact Match Accuracy**: All levels must match perfectly
   - **Per-Level Metrics**: Precision, Recall, F1 for each of state/district/block/village
   - **Micro-averaged F1**: Overall performance across all levels
   - **Level-by-level breakdown**: See exactly which administrative levels the model struggles with

4. **Better for Downstream Tasks**
   The structured output can be directly:
   - Parsed into a dictionary for programmatic access
   - Fed into geocoding APIs with proper hierarchy
   - Used for geographic analysis at specific administrative levels
   - Validated against known location hierarchies

#### Build Structured Location Column

In [ ]:
# Create structured human_annotated_location column from individual location fields
def build_structured_location(row):
    """Build structured location string from state, district, block, village columns."""
    parts = []
    
    if pd.notna(row.get('state')) and str(row.get('state')).strip():
        parts.append(f"state: {str(row['state']).strip()}")
    
    if pd.notna(row.get('district')) and str(row.get('district')).strip():
        parts.append(f"district: {str(row['district']).strip()}")
    
    if pd.notna(row.get('block')) and str(row.get('block')).strip():
        parts.append(f"block: {str(row['block']).strip()}")
    
    if pd.notna(row.get('village_name')) and str(row.get('village_name')).strip():
        parts.append(f"village: {str(row['village_name']).strip()}")
    
    return ', '.join(parts) if parts else ''

# Apply to create the human_annotated_location column
data['human_annotated_location'] = data.apply(build_structured_location, axis=1)

# Display sample to verify
print("Sample human-annotated locations:")
display(data[['state', 'district', 'block', 'village_name', 'human_annotated_location']].head(10))

#### Verify Data Structure

In [ ]:
# Display the first few rows to verify the data structure
print("Dataset shape:", data.shape)
print("\nFirst few rows:")
display(data.head())

### 2.3 Data Preprocessing and Filtering

#### Create Train/Validation/Test Splits

In [ ]:
from datasets import Dataset, DatasetDict

def prepare_data(df):
    # Updated prompt with structured output format
    inputs = [
        f"Extract location hierarchy from incident: {summary}\nFormat: state: <name>, district: <name>, block: <name>, village: <name>. Use exact format with labels. Omit missing levels."
        for summary in df['incident_summary']
    ]
    targets = df['human_annotated_location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 7854
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 982
    })
})


## 3. Model Setup and Training

**Padding Token Workflow:**
1. **During preprocessing** (Section 3.1): Padding tokens in labels are replaced with `-100`
2. **During training**: The loss function automatically ignores tokens marked as `-100`
3. **During decoding** (Section 5+): Filter out `-100` tokens before decoding: `[l for l in label if l != -100]`

**Optimization:**
- Training optimizes **cross-entropy loss** (`eval_loss`) on validation set
- Task-specific metrics (exact match, per-level F1) are computed **after training** for evaluation
- BLEU/ROUGE metrics are not used (better suited for text generation than structured extraction)

### 3.1 Initialize Tokenizer and Preprocessing

In [ ]:
from transformers import T5Tokenizer

# Initialize the tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# # Tokenization function
# def preprocess_function(examples):
#     # Tokenize the input and target text
#     model_inputs = tokenizer(
#         examples['input_text'],
#         max_length=512,
#         truncation=True,
#         padding="max_length"
#     )
#     labels = tokenizer(
#         examples['target_text'],
#         max_length=64,
#         truncation=True,
#         padding="max_length"
#     ).input_ids

#     # # Replace padding token id's in the labels with -100
#     # labels = [
#     #     [(l if l != tokenizer.pad_token_id else -100) for l in label]
#     #     for label in labels
#     # ]
#     model_inputs["labels"] = labels
#     return model_inputs

# def preprocess_function(examples):
#     inputs = examples['input_text']
#     targets = examples['target_text']

#     model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

#     # Tokenize targets with the `text_target` keyword argument
#     labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

#     model_inputs['labels'] = labels['input_ids']
#     return model_inputs

# # Apply tokenization
# tokenized_datasets = dataset.map(preprocess_function, batched=True)

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding='max_length'
    )

    # Tokenize targets and get input_ids
    labels = tokenizer(
        targets,
        max_length=64,
        truncation=True,
        padding='max_length'
    ).input_ids

    # Replace padding token id's in the labels with -100 so they are ignored by the loss
    labels = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels]

    model_inputs['labels'] = labels
    return model_inputs

# Apply tokenization to all splits
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Set format for PyTorch
tokenized_datasets.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

print("Tokenization complete!")
print(f"Train set: {len(tokenized_datasets['train'])} examples")
print(f"Validation set: {len(tokenized_datasets['validation'])} examples")
print(f"Test set: {len(tokenized_datasets['test'])} examples")


### 3.2 GPU Configuration

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### 3.3 Define Evaluation Functions

In [ ]:
def parse_structured_location(text):
    """Parse structured location string into a dictionary."""
    location_dict = {
        'state': None,
        'district': None,
        'block': None,
        'village': None
    }
    
    if not text or text.strip() == '':
        return location_dict
    
    # Split by comma and parse each part
    parts = [part.strip() for part in text.split(',')]
    for part in parts:
        if ':' in part:
            label, value = part.split(':', 1)
            label = label.strip().lower()
            value = value.strip()
            if label in location_dict:
                location_dict[label] = value
    
    return location_dict

# Test the parser with a sample
sample_location = "state: Chhattisgarh, district: Sukma, village: Nilamadgu"
parsed = parse_structured_location(sample_location)
print("Sample parsing test:")
print(f"Input: {sample_location}")
print(f"Parsed: {parsed}")

#### Parse Structured Location Output

In [ ]:
def compute_metrics(eval_pred):
    """Compute metrics for structured location predictions."""
    predictions, labels = eval_pred

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Initialize counters
    exact_matches = 0
    level_metrics = {
        'state': {'correct': 0, 'predicted': 0, 'total': 0},
        'district': {'correct': 0, 'predicted': 0, 'total': 0},
        'block': {'correct': 0, 'predicted': 0, 'total': 0},
        'village': {'correct': 0, 'predicted': 0, 'total': 0}
    }
    
    for pred_str, label_str in zip(decoded_preds, decoded_labels):
        pred_dict = parse_structured_location(pred_str)
        label_dict = parse_structured_location(label_str)
        
        # Exact match: all levels must be identical
        if pred_dict == label_dict:
            exact_matches += 1
        
        # Per-level metrics
        for level in ['state', 'district', 'block', 'village']:
            # Ground truth has this level
            if label_dict[level] is not None:
                level_metrics[level]['total'] += 1
                
                # Model predicted this level
                if pred_dict[level] is not None:
                    level_metrics[level]['predicted'] += 1
                    
                    # Check if it matches (case-insensitive)
                    if pred_dict[level].lower() == label_dict[level].lower():
                        level_metrics[level]['correct'] += 1
            
            # Model predicted this level when it shouldn't (false positive)
            elif pred_dict[level] is not None:
                level_metrics[level]['predicted'] += 1
    
    # Calculate metrics
    total_examples = len(decoded_preds)
    exact_match_accuracy = (exact_matches / total_examples) * 100 if total_examples > 0 else 0
    
    results = {
        'exact_match': exact_match_accuracy,
    }
    
    # Per-level F1 scores
    for level in ['state', 'district', 'block', 'village']:
        metrics = level_metrics[level]
        
        precision = (metrics['correct'] / metrics['predicted']) if metrics['predicted'] > 0 else 0
        recall = (metrics['correct'] / metrics['total']) if metrics['total'] > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        results[f'{level}_f1'] = f1 * 100
    
    # Overall micro-averaged F1 (across all levels)
    total_correct = sum(level_metrics[level]['correct'] for level in level_metrics)
    total_predicted = sum(level_metrics[level]['predicted'] for level in level_metrics)
    total_ground_truth = sum(level_metrics[level]['total'] for level in level_metrics)
    
    micro_precision = (total_correct / total_predicted) if total_predicted > 0 else 0
    micro_recall = (total_correct / total_ground_truth) if total_ground_truth > 0 else 0
    micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)) if (micro_precision + micro_recall) > 0 else 0
    
    results['micro_f1'] = micro_f1 * 100
    
    return results

#### Compute Training Metrics

In [ ]:
def compute_detailed_metrics(predictions, labels):
    """Compute detailed metrics for structured location predictions with full breakdown."""
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]
    
    # Initialize counters
    exact_matches = 0
    level_metrics = {
        'state': {'correct': 0, 'predicted': 0, 'total': 0},
        'district': {'correct': 0, 'predicted': 0, 'total': 0},
        'block': {'correct': 0, 'predicted': 0, 'total': 0},
        'village': {'correct': 0, 'predicted': 0, 'total': 0}
    }
    
    for pred_str, label_str in zip(decoded_preds, decoded_labels):
        pred_dict = parse_structured_location(pred_str)
        label_dict = parse_structured_location(label_str)
        
        # Exact match: all levels must be identical
        if pred_dict == label_dict:
            exact_matches += 1
        
        # Per-level metrics
        for level in ['state', 'district', 'block', 'village']:
            # Ground truth has this level
            if label_dict[level] is not None:
                level_metrics[level]['total'] += 1
                
                # Model predicted this level
                if pred_dict[level] is not None:
                    level_metrics[level]['predicted'] += 1
                    
                    # Check if it matches (case-insensitive)
                    if pred_dict[level].lower() == label_dict[level].lower():
                        level_metrics[level]['correct'] += 1
            
            # Model predicted this level when it shouldn't (false positive)
            elif pred_dict[level] is not None:
                level_metrics[level]['predicted'] += 1
    
    # Calculate metrics
    total_examples = len(decoded_preds)
    exact_match_accuracy = (exact_matches / total_examples) * 100 if total_examples > 0 else 0
    
    results = {
        'exact_match': exact_match_accuracy,
        'total_examples': total_examples
    }
    
    # Per-level precision, recall, F1
    print("\n" + "="*80)
    print("STRUCTURED LOCATION EXTRACTION METRICS")
    print("="*80)
    print(f"\nExact Match Accuracy: {exact_match_accuracy:.2f}%")
    print(f"Total Examples: {total_examples}\n")
    
    for level in ['state', 'district', 'block', 'village']:
        metrics = level_metrics[level]
        
        precision = (metrics['correct'] / metrics['predicted']) * 100 if metrics['predicted'] > 0 else 0
        recall = (metrics['correct'] / metrics['total']) * 100 if metrics['total'] > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        
        results[f'{level}_precision'] = precision
        results[f'{level}_recall'] = recall
        results[f'{level}_f1'] = f1
        results[f'{level}_support'] = metrics['total']
        
        print(f"{level.upper()} Level (support={metrics['total']}):")
        print(f"  Precision: {precision:.2f}% ({metrics['correct']}/{metrics['predicted']} predictions correct)")
        print(f"  Recall:    {recall:.2f}% ({metrics['correct']}/{metrics['total']} labels found)")
        print(f"  F1 Score:  {f1:.2f}%\n")
    
    # Overall micro-averaged F1 (across all levels)
    total_correct = sum(level_metrics[level]['correct'] for level in level_metrics)
    total_predicted = sum(level_metrics[level]['predicted'] for level in level_metrics)
    total_ground_truth = sum(level_metrics[level]['total'] for level in level_metrics)
    
    micro_precision = (total_correct / total_predicted) * 100 if total_predicted > 0 else 0
    micro_recall = (total_correct / total_ground_truth) * 100 if total_ground_truth > 0 else 0
    micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)) if (micro_precision + micro_recall) > 0 else 0
    
    results['micro_precision'] = micro_precision
    results['micro_recall'] = micro_recall
    results['micro_f1'] = micro_f1
    
    print("="*80)
    print("OVERALL (Micro-averaged across all levels):")
    print(f"  Precision: {micro_precision:.2f}%")
    print(f"  Recall:    {micro_recall:.2f}%")
    print(f"  F1 Score:  {micro_f1:.2f}%")
    print("="*80)
    
    return results

### 3.4 Initialize Model and Training Configuration

**Training Strategy:**
- **10 epochs maximum** with **early stopping** (patience=3 epochs)
- Training stops if validation loss doesn't improve for 3 consecutive epochs
- Prevents overfitting while allowing model to fully converge
- Saves only the 3 best checkpoints to conserve disk space
- Final model is the one with lowest validation loss

In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate

# Initialize the model
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.to(device)

# NOTE: BLEU/ROUGE metrics are not used during training
# The model optimizes cross-entropy loss (eval_loss)
# Task-specific structured location metrics (exact match, per-level F1) 
# are computed after training for evaluation purposes

# # UNUSED: Generic text similarity metrics (not suitable for structured extraction)
# # bleu = evaluate.load("bleu")
# 
# def compute_metrics(eval_pred):
#     """
#     UNUSED: This function is not called during training.
#     BLEU is designed for text generation (translation, summarization),
#     but we use structured location metrics (state F1, district F1, etc.) instead.
#     """
#     predictions, labels = eval_pred
#     decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
#     # Filter out -100 padding tokens before decoding
#     labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
# 
#     # Compute BLEU scores
#     bleu_result = bleu.compute(predictions=decoded_preds, references=[[ref] for ref in decoded_labels])
# 
#     metrics = {
#         "bleu": bleu_result["bleu"] * 100,
#     }
#     return metrics

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for checkpoints and logs
    eval_strategy="epoch",           # Evaluate at the end of every epoch
    learning_rate=5e-5,              # Learning rate
    per_device_train_batch_size=8,   # Batch size for training
    per_device_eval_batch_size=4,    # Batch size for evaluation
    num_train_epochs=10,             # Maximum number of training epochs
    weight_decay=0.01,               # Weight decay for regularization
    save_strategy="epoch",           # Save the model at each epoch
    save_total_limit=3,              # Keep only 3 best checkpoints (saves disk space)
    logging_dir="./logs",            # Directory for storing logs
    logging_steps=100,               # Log every 100 steps
    load_best_model_at_end=True,     # Load the best model at the end of training
    metric_for_best_model="eval_loss",  # Metric to monitor for best model
    greater_is_better=False,         # Lower eval_loss is better
    # Early stopping configuration
    early_stopping_patience=3,       # Stop if no improvement for 3 epochs
    early_stopping_threshold=0.001,  # Minimum change to qualify as improvement
    report_to='none'
)

# Initialize the Trainer with early stopping callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    # compute_metrics=compute_metrics
)


### 3.5 Train the Model

In [ ]:

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.089200,0.073098
2,0.074200,0.066961
3,0.071400,0.064558


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2946, training_loss=0.13490951619662966, metrics={'train_runtime': 3414.9844, 'train_samples_per_second': 6.9, 'train_steps_per_second': 0.863, 'total_flos': 1.434826581737472e+16, 'train_loss': 0.13490951619662966, 'epoch': 3.0})

### 3.6 Save Trained Model

In [ ]:
# Save the model and tokenizer to standardized location
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_dir = results_dir / "t5base_finetuned_model"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    print(f"✅ Model and tokenizer saved to: {model_dir}")
except NameError:
    # Fallback to Drive path for Colab
    model_path = '/content/drive/MyDrive/colab/satp-results/location-extraction/t5base_finetuned_model'
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"✅ Model and tokenizer saved to: {model_path}")

('/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/tokenizer_config.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/special_tokens_map.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/spiece.model',
 '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model/added_tokens.json')

## 4. Model Evaluation and Testing

### 4.1 Evaluate on Test Set

In [ ]:
# Evaluate the model on the test set
eval_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test Set Evaluation Results: {eval_results}")


Test Set Evaluation Results: {'eval_loss': 0.06318316608667374, 'eval_runtime': 47.1823, 'eval_samples_per_second': 20.813, 'eval_steps_per_second': 5.214, 'epoch': 3.0}


### 4.2 Define Prediction Function

In [ ]:
def predict_location(summary):
    # Prepare the input text with structured output prompt
    input_text = f"Extract location hierarchy from incident: {summary}\nFormat: state: <name>, district: <name>, block: <name>, village: <name>. Use exact format with labels. Omit missing levels."
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids

    # Generate predictions
    outputs = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)

    # Decode the output
    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_text

model.to('cpu')
# Example usage
incident_summary = "Three Communist Party of India-Maoist (CPI-Maoist) cadres, identified as Madvi Bhima (52), Madkam Hidma alias Sai Denga (33), and Paadam Ayate (24), surrendered in Sukma District in the Bastar division of Chhattisgarh on September 19, reports devdiscourse.com."
predicted_locations = predict_location(incident_summary)
print(f"Predicted Locations: {predicted_locations}")
print(f"\nParsed structure:")
parsed = parse_structured_location(predicted_locations)
for level, value in parsed.items():
    if value:
        print(f"  {level}: {value}")

Predicted Locations: chhattisgarh, sukma


## 5. Load Trained Model for Inference

### 5.1 Load Model and Tokenizer

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer from standardized location
try:
    results_dir = get_task_results_dir(TASK_NAME)
    model_path = results_dir / "t5base_finetuned_model"
    print(f"Loading model from: {model_path}")
except NameError:
    # Fallback to Drive path
    model_path = '/content/drive/MyDrive/colab/satp-results/location-extraction/t5base_finetuned_model'
    print(f"Loading model from: {model_path}")

model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'✅ Model loaded successfully')
print(f'Using device: {device}')

Using device: cuda
Using device: cuda


### 5.2 Prepare Test Dataset

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load the dataset from the URL
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info.csv"
data = pd.read_csv(url, dtype=str)

# Create structured human_annotated_location column from individual location fields
def build_structured_location(row):
    """Build structured location string from state, district, block, village columns."""
    parts = []
    
    if pd.notna(row.get('state')) and str(row.get('state')).strip():
        parts.append(f"state: {str(row['state']).strip()}")
    
    if pd.notna(row.get('district')) and str(row.get('district')).strip():
        parts.append(f"district: {str(row['district']).strip()}")
    
    if pd.notna(row.get('block')) and str(row.get('block')).strip():
        parts.append(f"block: {str(row['block']).strip()}")
    
    if pd.notna(row.get('village_name')) and str(row.get('village_name')).strip():
        parts.append(f"village: {str(row['village_name']).strip()}")
    
    return ', '.join(parts) if parts else ''

# Apply to create the human_annotated_location column
data['human_annotated_location'] = data.apply(build_structured_location, axis=1)

def prepare_data(df):
    # Updated prompt with structured output format
    inputs = [
        f"Extract location hierarchy from incident: {summary}\nFormat: state: <name>, district: <name>, block: <name>, village: <name>. Use exact format with labels. Omit missing levels."
        for summary in df['incident_summary']
    ]
    targets = df['human_annotated_location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
tokenized_dataset_test = dataset["test"].map(preprocess_function, batched=True)
# Set format for PyTorch
tokenized_dataset_test.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

Map:   0%|          | 0/982 [00:00<?, ? examples/s]

### 5.3 Define Evaluation Functions

In [ ]:
import numpy as np

def compute_metrics_for_evaluation(predictions, labels):
    """Compute metrics for structured location predictions - wrapper for standalone evaluation."""
    return compute_detailed_metrics(predictions, labels)

In [ ]:
import torch
from torch.utils.data import DataLoader

def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())  # Move labels to CPU

    # Compute detailed metrics with breakdown
    metrics = compute_detailed_metrics(all_predictions, all_labels)
    return metrics

### 5.4 Run Evaluation on Test Set

In [ ]:
from torch.utils.data import DataLoader

# Convert the test dataset to a PyTorch DataLoader
test_loader = DataLoader(tokenized_dataset_test, batch_size=8)


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


Test Set Metrics: {'exact_match': 40.52953156822811}


### 5.5 Save Evaluation Results

In [ ]:
# Save evaluation metrics to CSV
metrics_df = pd.DataFrame([metrics])
metrics_df['model'] = 'T5-base'
metrics_df['dataset'] = 'test'
metrics_df['task'] = 'location-extraction'

# Reorder columns for clarity
cols_order = ['task', 'model', 'dataset', 'exact_match', 'micro_f1', 'micro_precision', 'micro_recall',
              'state_f1', 'state_precision', 'state_recall', 'state_support',
              'district_f1', 'district_precision', 'district_recall', 'district_support',
              'block_f1', 'block_precision', 'block_recall', 'block_support',
              'village_f1', 'village_precision', 'village_recall', 'village_support',
              'total_examples']
metrics_df = metrics_df[[col for col in cols_order if col in metrics_df.columns]]

# Save using the standard utility function
try:
    saved_path = save_dataframe_csv(metrics_df, "location_extraction_test_metrics.csv", task_name=TASK_NAME)
    print(f"✅ Evaluation metrics saved to: {saved_path}")
except NameError:
    # Fallback if file_io utils not available
    output_path = "./location_extraction_test_metrics.csv"
    metrics_df.to_csv(output_path, index=False)
    print(f"✅ Evaluation metrics saved to: {output_path}")

display(metrics_df)

In [ ]:
# Generate and save predictions for the entire test set
print("Generating predictions for full test set...")

model.eval()
all_predictions = []
all_labels = []
all_inputs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        # Generate predictions
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=64
        )
        
        # Decode predictions and inputs
        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        inputs = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        
        # Decode labels: filter out -100 (padding tokens ignored during training)
        truths = [
            tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
            for label in labels
        ]
        
        all_predictions.extend(preds)
        all_labels.extend(truths)
        all_inputs.extend(inputs)

# Create predictions DataFrame
predictions_df = pd.DataFrame({
    'input_text': all_inputs,
    'ground_truth': all_labels,
    'prediction': all_predictions
})

# Parse structured outputs
predictions_df['gt_state'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('state'))
predictions_df['gt_district'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('district'))
predictions_df['gt_block'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('block'))
predictions_df['gt_village'] = predictions_df['ground_truth'].apply(lambda x: parse_structured_location(x).get('village'))

predictions_df['pred_state'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('state'))
predictions_df['pred_district'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('district'))
predictions_df['pred_block'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('block'))
predictions_df['pred_village'] = predictions_df['prediction'].apply(lambda x: parse_structured_location(x).get('village'))

# Add match indicators
predictions_df['exact_match'] = predictions_df.apply(
    lambda row: parse_structured_location(row['prediction']) == parse_structured_location(row['ground_truth']), 
    axis=1
)
predictions_df['state_match'] = predictions_df.apply(
    lambda row: (row['pred_state'] or '').lower() == (row['gt_state'] or '').lower() if row['gt_state'] else pd.NA,
    axis=1
)
predictions_df['district_match'] = predictions_df.apply(
    lambda row: (row['pred_district'] or '').lower() == (row['gt_district'] or '').lower() if row['gt_district'] else pd.NA,
    axis=1
)
predictions_df['block_match'] = predictions_df.apply(
    lambda row: (row['pred_block'] or '').lower() == (row['gt_block'] or '').lower() if row['gt_block'] else pd.NA,
    axis=1
)
predictions_df['village_match'] = predictions_df.apply(
    lambda row: (row['pred_village'] or '').lower() == (row['gt_village'] or '').lower() if row['gt_village'] else pd.NA,
    axis=1
)

# Save predictions
try:
    saved_path = save_dataframe_csv(predictions_df, "location_extraction_test_predictions.csv", task_name=TASK_NAME)
    print(f"✅ Test predictions saved to: {saved_path}")
    print(f"   Total predictions: {len(predictions_df)}")
    print(f"   Exact matches: {predictions_df['exact_match'].sum()} ({predictions_df['exact_match'].sum()/len(predictions_df)*100:.1f}%)")
except NameError:
    output_path = "./location_extraction_test_predictions.csv"
    predictions_df.to_csv(output_path, index=False)
    print(f"✅ Test predictions saved to: {output_path}")

display(predictions_df.head(10))

### 5.6 Verify Saved Results

In [ ]:
# List all files in the results directory
try:
    results_dir = get_task_results_dir(TASK_NAME)
    print(f"Results directory: {results_dir}\n")
    print("Saved files:")
    print("\n".join(sorted(os.listdir(results_dir))))
except NameError:
    print("Results saved in current directory:")
    import glob
    csv_files = glob.glob("location_extraction*.csv")
    if csv_files:
        print("\n".join(sorted(csv_files)))
    else:
        print("No result files found yet.")

In [ ]:
# Quick view: Show 10 random examples with side-by-side comparison
model.eval()
batch = next(iter(test_loader))

input_ids = batch["input_ids"][:10].to(device)
attention_mask = batch["attention_mask"][:10].to(device)
labels = batch["labels"][:10]

# Generate predictions
with torch.no_grad():
    outputs = model.generate(input_ids, attention_mask=attention_mask, max_length=64)

# Decode
predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
ground_truth = [
    tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
    for label in labels
]

# Display side by side with structured parsing
print("\nSample Predictions vs Ground Truth (Structured Format):")
print("="*100)
for i, (pred, truth) in enumerate(zip(predictions, ground_truth), 1):
    pred_dict = parse_structured_location(pred)
    truth_dict = parse_structured_location(truth)
    match = "✓ EXACT MATCH" if pred_dict == truth_dict else "✗ PARTIAL/NO MATCH"
    
    print(f"\n[{i}] {match}")
    print(f"Predicted:    '{pred}'")
    print(f"Ground Truth: '{truth}'")
    
    if pred_dict != truth_dict:
        # Show level-by-level comparison
        print("  Level breakdown:")
        for level in ['state', 'district', 'block', 'village']:
            pred_val = pred_dict[level] or "—"
            truth_val = truth_dict[level] or "—"
            if pred_val == truth_val:
                print(f"    {level:10}: ✓ {truth_val}")
            else:
                print(f"    {level:10}: ✗ predicted='{pred_val}', actual='{truth_val}'")

## 6. Google API Geocoding Integration

### 6.1 Setup Google API Client

# **INFERENCE**

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer
model_path = '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/t5base_finetuned_model'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'Using device: {device}')


Using device: cuda
Using device: cuda


In [ ]:
def extract_locations(summary):
    # Prepare the input text with structured output prompt
    input_text = f"Extract location hierarchy from incident: {summary}\nFormat: state: <name>, district: <name>, block: <name>, village: <name>. Use exact format with labels. Omit missing levels."
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids.to(device)  # Move input_ids to the device

    # Generate predictions
    outputs = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)

    # Decode the output
    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_text

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

# Google Maps API key
API_KEY = userdata.get('googlemapsAPI')
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"


# Updated function to get location details including latitude and longitude
def get_location_details(locations):
    """Given a list of location names, constructs a query, calls the Google Geocoding API,
    and returns state, district, subdistrict, town/village, and latitude/longitude of the most specific level."""
    #query = ', '.join(locations)
    params = {
        'address': locations,
        'key': API_KEY,
        'components': 'country:IN'
    }
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        print(f"Geocoding API error: {data['status']}")
        return None

    # Initialize components
    state = district = subdistrict = town_village = None
    latitude = longitude = None
    found_level = None  # Keep track of the most specific level found

    # Desired levels in order of specificity
    levels = ['locality', 'administrative_area_level_3', 'administrative_area_level_2']

    # Iterate over results to find the most specific level
    for result in data.get('results', []):
        temp_state = temp_district = temp_subdistrict = temp_town_village = None
        address_components = result['address_components']

        # Map address components
        for component in address_components:
            types = component['types']
            if 'administrative_area_level_1' in types:
                temp_state = component['long_name']
            elif 'administrative_area_level_2' in types:
                temp_district = component['long_name']
            elif 'administrative_area_level_3' in types:
                temp_subdistrict = component['long_name']
            elif 'locality' in types:
                temp_town_village = component['long_name']
            elif 'sublocality' in types and not temp_town_village:
                temp_town_village = component['long_name']

        # Determine the most specific level in this result
        if temp_town_village and found_level not in ['town_village']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = temp_town_village
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'town_village'
        elif temp_subdistrict and found_level not in ['town_village', 'subdistrict']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'subdistrict'
        elif temp_district and found_level not in ['town_village', 'subdistrict', 'district']:
            state = temp_state
            district = temp_district
            subdistrict = None
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'district'

        # Break the loop if the most specific level is found
        if found_level == 'town_village':
            break

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village,
        'latitude': latitude,
        'longitude': longitude
    }

In [ ]:
def parse_structured_to_address(structured_location):
    """
    Parse T5 structured output and build a proper address string for Google API.
    Input: "state: Chhattisgarh, district: Sukma, village: Nilamadgu"
    Output: "Nilamadgu, Sukma, Chhattisgarh, India"
    """
    if not structured_location or structured_location.strip() == '':
        return None
    
    parsed = parse_structured_location(structured_location)
    
    # Build address from most specific to least specific (reverse order)
    parts = []
    for level in ['village', 'block', 'district', 'state']:
        if parsed[level]:
            parts.append(parsed[level])
    
    if parts:
        parts.append('India')
        return ', '.join(parts)
    return None


def build_address_from_columns(row):
    """
    Build a structured address string directly from human annotation columns.
    This reflects reality: SATP always categorizes by state, so humans always knew the state.
    We correct for Telangana split and ALWAYS include the state.
    """
    parts = []
    
    # Correct state for Telangana (if needed)
    corrected_state = correct_state_for_telangana(row)
    
    # Add from most specific to least specific
    if pd.notna(row.get('village_name')) and str(row.get('village_name')).strip():
        parts.append(str(row['village_name']).strip())
    
    if pd.notna(row.get('block')) and str(row.get('block')).strip():
        parts.append(str(row['block']).strip())
    
    if pd.notna(row.get('district')) and str(row.get('district')).strip():
        parts.append(str(row['district']).strip())
    
    # ALWAYS include state (since SATP categorizes by state)
    if corrected_state and str(corrected_state).strip():
        parts.append(str(corrected_state).strip())
    
    # ALWAYS include country
    parts.append('India')
    return ', '.join(parts) if parts else None


# Test the functions
print("Testing address building functions:\n")

# Test 1: Parse T5 structured output
sample_t5_output = "state: Chhattisgarh, district: Sukma, village: Nilamadgu"
address_from_t5 = parse_structured_to_address(sample_t5_output)
print(f"T5 Output: {sample_t5_output}")
print(f"Address for API: {address_from_t5}\n")

# Test 2: Build from columns (simulate a row)
sample_row = pd.Series({
    'state': 'Chhattisgarh',
    'district': 'Sukma',
    'block': 'Kistaram',
    'village_name': 'Nilamadgu'
})
address_from_columns = build_address_from_columns(sample_row)
print(f"Human annotations: state={sample_row['state']}, district={sample_row['district']}, block={sample_row['block']}, village={sample_row['village_name']}")
print(f"Address for API: {address_from_columns}")

## Important Implementation Details

### 1. State is Always Known
Since SATP categorizes incidents by state, the humans always had state information available. Therefore, for the **Human Annotations → Google API** approach, we **always include the state** in the geocoding query. This reflects the real-world scenario and makes for a fairer comparison.

### 2. Telangana Split (June 2, 2014)
Telangana was carved out of Andhra Pradesh on June 2, 2014. We handle this by:
- Maintaining a list of districts that became part of Telangana
- For incidents after June 2, 2014 in those districts, we use "Telangana" as the state
- For incidents before that date, we use "Andhra Pradesh"
- This ensures correct geocoding regardless of how the state was originally coded in SATP

### 3. Address Format for Google API
Both approaches now build addresses as:
```
village, block, district, state, India
```

Where:
- **LLM approach**: Extracts all levels from text (may miss state)
- **Human approach**: Always includes state (since it was known), uses human-annotated sub-state levels

In [ ]:
# Telangana districts that split from Andhra Pradesh on June 2, 2014
TELANGANA_DISTRICTS = {
    'Adilabad', 'Bhadradri Kothagudem', 'Hyderabad', 'Jagtial', 'Jangaon',
    'Jayashankar Bhupalpally', 'Jogulamba Gadwal', 'Kamareddy', 'Karimnagar',
    'Khammam', 'Komaram Bheem', 'Mahabubabad', 'Mahabubnagar', 'Mancherial',
    'Medak', 'Medchal–Malkajgiri', 'Mulugu', 'Nagarkurnool', 'Nalgonda',
    'Narayanpet', 'Nirmal', 'Nizamabad', 'Peddapalli', 'Rajanna Sircilla',
    'Rangareddy', 'Sangareddy', 'Siddipet', 'Suryapet', 'Vikarabad',
    'Wanaparthy', 'Warangal Rural', 'Warangal Urban', 'Yadadri Bhuvanagiri'
}

def correct_state_for_telangana(row):
    """
    Correct state name for Telangana districts after June 2, 2014.
    Returns the corrected state name.
    """
    state = row.get('state', '')
    district = row.get('district', '')
    
    # Parse date if available
    date_str = row.get('date')  # Adjust column name as needed
    if pd.notna(date_str):
        try:
            from datetime import datetime
            # Try to parse the date - adjust format as needed
            if isinstance(date_str, str):
                # Try common formats
                for fmt in ['%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y', '%Y/%m/%d']:
                    try:
                        incident_date = datetime.strptime(date_str, fmt)
                        break
                    except:
                        continue
                else:
                    incident_date = None
            else:
                incident_date = date_str  # Already a datetime
                
            if incident_date:
                telangana_formation = datetime(2014, 6, 2)
                
                # If incident is after Telangana formation and district is in Telangana
                if incident_date >= telangana_formation:
                    if district in TELANGANA_DISTRICTS:
                        return 'Telangana'
                    elif state == 'Telangana':
                        return 'Telangana'
                # If incident is before formation, use Andhra Pradesh for those districts
                elif district in TELANGANA_DISTRICTS and state == 'Telangana':
                    return 'Andhra Pradesh'
        except:
            pass  # If date parsing fails, use original state
    
    return state

# Test the function
print("Testing Telangana correction:\n")
test_cases = [
    {'state': 'Andhra Pradesh', 'district': 'Hyderabad', 'date': '2015-01-01'},  # Should be Telangana
    {'state': 'Andhra Pradesh', 'district': 'Nalgonda', 'date': '2013-01-01'},   # Should stay AP
    {'state': 'Telangana', 'district': 'Hyderabad', 'date': '2015-01-01'},       # Should stay Telangana
]

for i, test in enumerate(test_cases, 1):
    corrected_state = correct_state_for_telangana(pd.Series(test))
    print(f"Test {i}: {test['district']}, {test['state']} ({test['date']}) → {corrected_state}")

In [ ]:
from sklearn.model_selection import train_test_split

# Load the dataset from the URL
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/satp_location_matched_lat_long.csv"

data = pd.read_csv(url, dtype=str)
# Split the dataframe into train, validation, and test sets (80% train, 10% val, 10% test)
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

# Display the shapes of the split dataframes to verify
print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

Train set shape: (7854, 11)
Validation set shape: (982, 11)
Test set shape: (982, 11)


In [ ]:
# Load the incident data
satp_data = test_df.copy()

# Columns for LLM extraction → API geocoding
satp_data['llm_extracted_locations'] = None
satp_data['llm_extracted_address'] = None
satp_data['llm_extracted_state'] = None
satp_data['llm_extracted_district'] = None
satp_data['llm_extracted_subdistrict'] = None
satp_data['llm_extracted_town_village'] = None
satp_data['llm_extracted_latitude'] = None
satp_data['llm_extracted_longitude'] = None

# Columns for Human annotations → API geocoding (gold standard)
satp_data['human_address'] = None
satp_data['human_geocoded_state'] = None
satp_data['human_geocoded_district'] = None
satp_data['human_geocoded_subdistrict'] = None
satp_data['human_geocoded_town_village'] = None
satp_data['human_geocoded_latitude'] = None
satp_data['human_geocoded_longitude'] = None

In [ ]:
# Process each summary in the data
for index, row in satp_data.iterrows():
    summary = row['incident_summary']
    
    # ===== APPROACH 1: LLM Extraction → Google API =====
    # Extract structured locations using T5 model
    llm_structured_output = extract_locations(summary)
    satp_data.at[index, 'llm_extracted_locations'] = llm_structured_output
    
    # Convert structured output to address string for Google API
    llm_address = parse_structured_to_address(llm_structured_output)
    satp_data.at[index, 'llm_extracted_address'] = llm_address
    
    if llm_address:
        location_details = get_location_details(llm_address)
        if location_details:
            satp_data.at[index, 'llm_extracted_state'] = location_details.get('state')
            satp_data.at[index, 'llm_extracted_district'] = location_details.get('district')
            satp_data.at[index, 'llm_extracted_subdistrict'] = location_details.get('subdistrict')
            satp_data.at[index, 'llm_extracted_town_village'] = location_details.get('town_village')
            satp_data.at[index, 'llm_extracted_latitude'] = location_details.get('latitude')
            satp_data.at[index, 'llm_extracted_longitude'] = location_details.get('longitude')
    
    # ===== APPROACH 2: Human Annotations → Google API (Gold Standard) =====
    # Build address directly from human annotation columns
    human_address = build_address_from_columns(row)
    satp_data.at[index, 'human_address'] = human_address
    
    if human_address:
        location_details = get_location_details(human_address)
        if location_details:
            satp_data.at[index, 'human_geocoded_state'] = location_details.get('state')
            satp_data.at[index, 'human_geocoded_district'] = location_details.get('district')
            satp_data.at[index, 'human_geocoded_subdistrict'] = location_details.get('subdistrict')
            satp_data.at[index, 'human_geocoded_town_village'] = location_details.get('town_village')
            satp_data.at[index, 'human_geocoded_latitude'] = location_details.get('latitude')
            satp_data.at[index, 'human_geocoded_longitude'] = location_details.get('longitude')
    
    if index % 10 == 0:
        print(f"Processed {index + 1}/{len(satp_data)} rows...")

Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS


In [ ]:
# Define the path to save the CSV file with both approaches
output_path = '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/satp_comparison_llm_vs_human_geocoding.csv'

# Save the complete DataFrame with all approaches
satp_data_complete.to_csv(output_path, index=False)

print(f"DataFrame saved successfully to {output_path}")
print(f"\nSaved {len(satp_data_complete)} rows with complete data for all approaches")
print("\nColumns saved:")
print("- Manual baseline: latitude, longitude")
print("- LLM approach: llm_extracted_locations, llm_extracted_latitude, llm_extracted_longitude, llm_distance_km")
print("- Human approach: human_address, human_geocoded_latitude, human_geocoded_longitude, human_distance_km")

DataFrame saved successfully to /content/drive/MyDrive/SATP_data/location_context_extraction_exp/satp_data_with_extracted_locations_lat_long.csv


In [ ]:
# Display sample results comparing both approaches
print("Sample comparison of LLM extraction vs Human annotation geocoding:\n")
sample_cols = [
    'incident_summary',
    'state', 'district', 'village_name',  # Original human annotations
    'llm_extracted_locations', 'llm_extracted_address',  # LLM approach
    'human_address',  # Human approach
    'llm_extracted_latitude', 'llm_extracted_longitude',
    'human_geocoded_latitude', 'human_geocoded_longitude'
]
display(satp_data[sample_cols].head(5))

## 7. Comparative Evaluation: LLM Extraction vs Human Annotations

This section compares two geocoding approaches:
1. **LLM Approach**: T5 model extracts locations → Google API geocodes
2. **Human Approach**: Human annotations → Google API geocodes (Gold Standard)

### Evaluation Framework

We measure accuracy by calculating the geodesic distance between predicted and ground truth coordinates. This allows us to:
- Quantify location extraction accuracy in kilometers
- Compare performance at different geographic granularities (state, district, block, village)
- Understand where the model excels vs. struggles

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

# Load the DataFrame from the saved CSV file
input_path = '/content/drive/MyDrive/SATP_data/location_context_extraction_exp/satp_data_with_extracted_locations_lat_long.csv'
satp_data = pd.read_csv(input_path)

# Display the first few rows to verify the data
display(satp_data.head())

,incident_number,state,district,block,village_name,other_areas,constituency,incident_summary,Matched_Location,latitude,longitude,extracted_state,extracted_district,extracted_subdistrict,extracted_town_village,extracted_latitude,extracted_longitude,extracted_locations
0,1406200901,Odisha,Kandhmal,Daringbadi,Sonepur,NaN,G. Udayagiri,"Four persons, including the husbands of two el...","kandhmal, daringbadi, sonepur",83.8950,20.8480,Odisha,Southern Division,Kandhamal,Sonpur,19.781595,84.024673,"kandhmal, sonepur"
1,204131401,Bihar,Lakhisarai,NaN,NaN,NaN,NaN,CPI-Maoist cadres blew off a part of the railw...,lakhisarai,NaN,NaN,Bihar,Patna Division,Patna,Patna,25.594095,85.137564,"lakhisarai, howrah, patna"
2,1412021101,Odisha,Koraput,Baipariguda,Mathapada,Kalia Attalla,Kotpad (st),The CPI-Maoist cadres set ablaze two cell phon...,"koraput, baipariguda, mathapada, kalia attalla",82.3072,18.7858,Odisha,Southern Division,Koraput,Boipariguda,18.751105,82.433295,"koraput, baipariguda, kalia attalla"
3,1409041401,Odisha,Balangir,Khaparkhol,Khuripani,Saapmund in the Gandharmardan reserve forest,Patnagarh,"In an anti-Maoist operation, the 189 Battalion...","odisha, balangir, khuripani",82.8627,20.7504,Odisha,Northern Division,Balangir,Khuripani,20.821604,82.829055,"odisha, balangir, khuripani, saapmund"
4,307011001,Chhattisgarh,Narayanpur,NaN,NaN,NaN,NaN,Three civilians were reportedly killed by Maoi...,narayanpur,NaN,NaN,Chhattisgarh,Bastar Division,Narayanpur,Narayanpur,19.719557,81.247197,narayanpur


## Data preparation

### Subtask:
Ensure both actual and predicted latitude and longitude columns are in a suitable format (e.g., numeric). Handle any missing or invalid values.


**Reasoning**:
Convert latitude and longitude columns to numeric, handle missing values by dropping rows with any missing values in the relevant columns, and verify the result.



In [ ]:
# Convert latitude and longitude columns to numeric, coercing errors
satp_data['latitude'] = pd.to_numeric(satp_data['latitude'], errors='coerce')
satp_data['longitude'] = pd.to_numeric(satp_data['longitude'], errors='coerce')
satp_data['llm_extracted_latitude'] = pd.to_numeric(satp_data['llm_extracted_latitude'], errors='coerce')
satp_data['llm_extracted_longitude'] = pd.to_numeric(satp_data['llm_extracted_longitude'], errors='coerce')
satp_data['human_geocoded_latitude'] = pd.to_numeric(satp_data['human_geocoded_latitude'], errors='coerce')
satp_data['human_geocoded_longitude'] = pd.to_numeric(satp_data['human_geocoded_longitude'], errors='coerce')

# Identify and count missing values before dropping
print("Missing values in manual baseline coordinates:")
print(satp_data[['latitude', 'longitude']].isnull().sum())

print("\nMissing values in LLM extraction → API geocoding:")
print(satp_data[['llm_extracted_latitude', 'llm_extracted_longitude']].isnull().sum())

print("\nMissing values in Human annotations → API geocoding:")
print(satp_data[['human_geocoded_latitude', 'human_geocoded_longitude']].isnull().sum())

# Create a clean version for comparison (rows where we have all three coordinate sets)
satp_data_complete = satp_data.dropna(subset=[
    'latitude', 'longitude',  # Manual baseline
    'llm_extracted_latitude', 'llm_extracted_longitude',  # LLM approach
    'human_geocoded_latitude', 'human_geocoded_longitude'  # Human approach
]).copy()

print(f"\nRows with complete data for all three approaches: {len(satp_data_complete)}/{len(satp_data)}")

Missing values before dropping:
latitude               0
longitude              0
extracted_latitude     0
extracted_longitude    0
dtype: int64

Missing values after dropping:
latitude               0
longitude              0
extracted_latitude     0
extracted_longitude    0
dtype: int64

Shape of DataFrame after dropping rows: (725, 18)


### 7.1 Calculate Distance Metrics

In [ ]:
!pip install haversine

#### Install Distance Calculation Library

#### Compute Haversine Distances

In [ ]:
from haversine import haversine, Unit

# Define a function to calculate the Haversine distance for LLM extraction
def calculate_llm_distance(row):
    """Calculate distance between manual baseline and LLM extraction → API geocoding"""
    manual_coords = (row['latitude'], row['longitude'])
    llm_coords = (row['llm_extracted_latitude'], row['llm_extracted_longitude'])
    if pd.isna(manual_coords[0]) or pd.isna(manual_coords[1]) or pd.isna(llm_coords[0]) or pd.isna(llm_coords[1]):
        return None
    return haversine(manual_coords, llm_coords, unit=Unit.KILOMETERS)

# Define a function to calculate the Haversine distance for human annotations
def calculate_human_distance(row):
    """Calculate distance between manual baseline and Human annotations → API geocoding"""
    manual_coords = (row['latitude'], row['longitude'])
    human_coords = (row['human_geocoded_latitude'], row['human_geocoded_longitude'])
    if pd.isna(manual_coords[0]) or pd.isna(manual_coords[1]) or pd.isna(human_coords[0]) or pd.isna(human_coords[1]):
        return None
    return haversine(manual_coords, human_coords, unit=Unit.KILOMETERS)

# Apply the functions to create distance columns
satp_data_complete['llm_distance_km'] = satp_data_complete.apply(calculate_llm_distance, axis=1)
satp_data_complete['human_distance_km'] = satp_data_complete.apply(calculate_human_distance, axis=1)

# Display the first few rows with the new columns
print("Sample distances (km from manual baseline):\n")
display(satp_data_complete[['incident_number', 'llm_distance_km', 'human_distance_km']].head(10))

,incident_number,state,district,block,village_name,other_areas,constituency,incident_summary,Matched_Location,latitude,longitude,extracted_state,extracted_district,extracted_subdistrict,extracted_town_village,extracted_latitude,extracted_longitude,extracted_locations,distance_km
8066,1406200901,Odisha,Kandhmal,Daringbadi,Sonepur,NaN,G. Udayagiri,"Four persons, including the husbands of two el...","kandhmal, daringbadi, sonepur",20.848000,83.895000,Odisha,Southern Division,Kandhamal,Sonpur,19.781595,84.024673,"kandhmal, sonepur",119.347457
8712,1412021101,Odisha,Koraput,Baipariguda,Mathapada,Kalia Attalla,Kotpad (st),The CPI-Maoist cadres set ablaze two cell phon...,"koraput, baipariguda, mathapada, kalia attalla",18.785800,82.307200,Odisha,Southern Division,Koraput,Boipariguda,18.751105,82.433295,"koraput, baipariguda, kalia attalla",13.824810
8365,1409041401,Odisha,Balangir,Khaparkhol,Khuripani,Saapmund in the Gandharmardan reserve forest,Patnagarh,"In an anti-Maoist operation, the 189 Battalion...","odisha, balangir, khuripani",20.750400,82.862700,Odisha,Northern Division,Balangir,Khuripani,20.821604,82.829055,"odisha, balangir, khuripani, saapmund",8.655664
6761,904120901,Karnataka,Chikmagalur,Koppa,Megunda,Baisigadde,Sringeri,Police arrested three sympathizers of the CPI-...,"chikmagalur, koppa, megunda, baisigadde, sringeri",13.337800,75.322300,Karnataka,Mysore Division,Chikkamagaluru,Megunda,13.337837,75.322268,"chikmagalur, kachige, megunda revenue division",0.005360
1945,201221401,Bihar,Patna,NaN,NaN,Patna Medical College,NaN,The Patna city Police of Bihar arrested a CPI-...,"bihar, patna, patna medical college",25.621017,85.160416,Bihar,Patna Division,Patna,Patna,25.594095,85.137564,"bihar, patna",3.769916


### 7.2 Accuracy Statistics and Comparisons

In [ ]:
import numpy as np

print("="*80)
print("GEOCODING ACCURACY COMPARISON")
print("="*80)
print(f"\nBaseline: Manual geocoding (human looked up in Google)")
print(f"Total incidents with complete data: {len(satp_data_complete)}\n")

# Define distance thresholds
thresholds = [1, 5, 10, 50, 100]

# ===== LLM EXTRACTION → API GEOCODING =====
print("="*80)
print("APPROACH 1: LLM Extraction → Google API")
print("="*80)
mean_llm = satp_data_complete['llm_distance_km'].mean()
median_llm = satp_data_complete['llm_distance_km'].median()
std_llm = satp_data_complete['llm_distance_km'].std()

print(f"Mean distance from manual baseline: {mean_llm:.2f} km")
print(f"Median distance: {median_llm:.2f} km")
print(f"Standard deviation: {std_llm:.2f} km")

print("\nPercentage within distance thresholds:")
for threshold in thresholds:
    within_threshold = satp_data_complete[satp_data_complete['llm_distance_km'] <= threshold].shape[0]
    percentage = (within_threshold / len(satp_data_complete)) * 100
    print(f"  <= {threshold} km: {percentage:.2f}%")

# ===== HUMAN ANNOTATIONS → API GEOCODING =====
print("\n" + "="*80)
print("APPROACH 2: Human Annotations → Google API (Gold Standard)")
print("="*80)
mean_human = satp_data_complete['human_distance_km'].mean()
median_human = satp_data_complete['human_distance_km'].median()
std_human = satp_data_complete['human_distance_km'].std()

print(f"Mean distance from manual baseline: {mean_human:.2f} km")
print(f"Median distance: {median_human:.2f} km")
print(f"Standard deviation: {std_human:.2f} km")

print("\nPercentage within distance thresholds:")
for threshold in thresholds:
    within_threshold = satp_data_complete[satp_data_complete['human_distance_km'] <= threshold].shape[0]
    percentage = (within_threshold / len(satp_data_complete)) * 100
    print(f"  <= {threshold} km: {percentage:.2f}%")

# ===== COMPARISON =====
print("\n" + "="*80)
print("COMPARISON: Performance Gap")
print("="*80)
print(f"Difference in mean distance: {mean_llm - mean_human:.2f} km")
print(f"Difference in median distance: {median_llm - median_human:.2f} km")

print("\nPerformance gap at each threshold:")
for threshold in thresholds:
    llm_pct = (satp_data_complete[satp_data_complete['llm_distance_km'] <= threshold].shape[0] / len(satp_data_complete)) * 100
    human_pct = (satp_data_complete[satp_data_complete['human_distance_km'] <= threshold].shape[0] / len(satp_data_complete)) * 100
    gap = human_pct - llm_pct
    print(f"  <= {threshold} km: {gap:+.2f} percentage points (Human is {'better' if gap > 0 else 'worse'})")

print("\n" + "="*80)

Mean distance: 82.09 km
Median distance: 18.62 km
Standard deviation of distance: 249.91 km

Percentage of incidents within distance thresholds:
  <= 1 km: 32.97%
  <= 5 km: 38.62%
  <= 10 km: 42.90%
  <= 50 km: 68.41%
  <= 100 km: 81.52%


In [ ]:
import matplotlib.pyplot as plt

# Create visualizations comparing both approaches
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Histogram comparison
axes[0, 0].hist(satp_data_complete['llm_distance_km'], bins=50, alpha=0.6, label='LLM Extraction', color='blue')
axes[0, 0].hist(satp_data_complete['human_distance_km'], bins=50, alpha=0.6, label='Human Annotations', color='green')
axes[0, 0].set_xlabel('Distance from Manual Baseline (km)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Geocoding Errors')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 100)  # Focus on first 100km

# 2. Box plot comparison
data_to_plot = [
    satp_data_complete['llm_distance_km'].values,
    satp_data_complete['human_distance_km'].values
]
axes[0, 1].boxplot(data_to_plot, labels=['LLM Extraction', 'Human Annotations'])
axes[0, 1].set_ylabel('Distance (km)')
axes[0, 1].set_title('Error Distribution Comparison')
axes[0, 1].set_yscale('log')  # Log scale for better visualization
axes[0, 1].grid(True, alpha=0.3)

# 3. Cumulative percentage within thresholds
thresholds = range(0, 101, 5)
llm_cumulative = []
human_cumulative = []

for threshold in thresholds:
    llm_pct = (satp_data_complete['llm_distance_km'] <= threshold).sum() / len(satp_data_complete) * 100
    human_pct = (satp_data_complete['human_distance_km'] <= threshold).sum() / len(satp_data_complete) * 100
    llm_cumulative.append(llm_pct)
    human_cumulative.append(human_pct)

axes[1, 0].plot(thresholds, llm_cumulative, label='LLM Extraction', marker='o', markersize=3, color='blue')
axes[1, 0].plot(thresholds, human_cumulative, label='Human Annotations', marker='s', markersize=3, color='green')
axes[1, 0].set_xlabel('Distance Threshold (km)')
axes[1, 0].set_ylabel('Cumulative % within Threshold')
axes[1, 0].set_title('Cumulative Accuracy by Distance Threshold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Scatter plot: Direct comparison
axes[1, 1].scatter(satp_data_complete['human_distance_km'], 
                   satp_data_complete['llm_distance_km'], 
                   alpha=0.5, s=20)
axes[1, 1].plot([0, 100], [0, 100], 'r--', label='Equal performance')
axes[1, 1].set_xlabel('Human Annotations Distance (km)')
axes[1, 1].set_ylabel('LLM Extraction Distance (km)')
axes[1, 1].set_title('Direct Comparison (below red line = LLM better)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim(0, 100)
axes[1, 1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("\nVisualization Notes:")
print("- Blue = LLM Extraction → Google API")
print("- Green = Human Annotations → Google API (ceiling performance)")
print("- The gap between them shows the cost of extraction errors")

### 7.3 Analysis by Geographic Granularity

In [ ]:
# Determine the original location level for each row
# Priority: village_name > block > district > state
def determine_location_level(row):
    """Determine the most specific location level available for this row."""
    if pd.notna(row.get('village_name')) and str(row.get('village_name')).strip() != '':
        return 'village'
    elif pd.notna(row.get('block')) and str(row.get('block')).strip() != '':
        return 'block'
    elif pd.notna(row.get('district')) and str(row.get('district')).strip() != '':
        return 'district'
    elif pd.notna(row.get('state')) and str(row.get('state')).strip() != '':
        return 'state'
    else:
        return 'unknown'

# Add location level column
satp_data['original_location_level'] = satp_data.apply(determine_location_level, axis=1)

# Display the distribution of location levels
print("Distribution of original location levels:")
print(satp_data['original_location_level'].value_counts())
print(f"\nTotal rows: {len(satp_data)}")


In [ ]:
# Calculate metrics for each location level
location_levels = ['village', 'block', 'district', 'state', 'unknown']
results_by_level = {}

for level in location_levels:
    level_data = satp_data[satp_data['original_location_level'] == level]
    
    if len(level_data) == 0:
        continue
    
    results_by_level[level] = {
        'count': len(level_data),
        'mean_distance': level_data['distance_km'].mean(),
        'median_distance': level_data['distance_km'].median(),
        'std_distance': level_data['distance_km'].std(),
        'min_distance': level_data['distance_km'].min(),
        'max_distance': level_data['distance_km'].max(),
    }
    
    # Calculate percentages within thresholds
    thresholds = [1, 5, 10, 50, 100]
    for threshold in thresholds:
        within_threshold = len(level_data[level_data['distance_km'] <= threshold])
        percentage = (within_threshold / len(level_data)) * 100
        results_by_level[level][f'pct_within_{threshold}km'] = percentage

# Create a summary dataframe
summary_df = pd.DataFrame(results_by_level).T
summary_df = summary_df.round(2)

print("Metrics by Original Location Level:")
print("=" * 80)
display(summary_df)


In [ ]:
# Print detailed metrics for each level
print("\nDetailed Metrics by Location Level:")
print("=" * 80)

for level in ['village', 'block', 'district', 'state']:
    if level not in results_by_level:
        continue
    
    level_data = satp_data[satp_data['original_location_level'] == level]
    metrics = results_by_level[level]
    
    print(f"\n{level.upper()}-LEVEL COORDINATES (n={metrics['count']}):")
    print(f"  Mean distance: {metrics['mean_distance']:.2f} km")
    print(f"  Median distance: {metrics['median_distance']:.2f} km")
    print(f"  Standard deviation: {metrics['std_distance']:.2f} km")
    print(f"  Range: {metrics['min_distance']:.2f} - {metrics['max_distance']:.2f} km")
    print(f"\n  Percentage within thresholds:")
    thresholds = [1, 5, 10, 50, 100]
    for threshold in thresholds:
        print(f"    <= {threshold} km: {metrics[f'pct_within_{threshold}km']:.2f}%")


In [ ]:
# Optional: Create visualizations comparing distributions across levels
import matplotlib.pyplot as plt

# Box plot of distances by location level
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot
levels_to_plot = [level for level in ['village', 'block', 'district', 'state'] 
                  if level in results_by_level]
data_to_plot = [satp_data[satp_data['original_location_level'] == level]['distance_km'].values 
                for level in levels_to_plot]

axes[0].boxplot(data_to_plot, labels=levels_to_plot)
axes[0].set_ylabel('Distance (km)')
axes[0].set_xlabel('Original Location Level')
axes[0].set_title('Distribution of Distance Errors by Location Level')
axes[0].set_yscale('log')  # Use log scale for better visualization
axes[0].grid(True, alpha=0.3)

# Bar chart of mean distances
means = [results_by_level[level]['mean_distance'] for level in levels_to_plot]
axes[1].bar(levels_to_plot, means)
axes[1].set_ylabel('Mean Distance (km)')
axes[1].set_xlabel('Original Location Level')
axes[1].set_title('Mean Distance Error by Location Level')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nNote: Log scale used for box plot to better visualize differences across levels.")
